# Upwork Freelance Jobs: Data Profiling & Cleaning

This notebook aims to clean, profile, and categorize a dataset of Upwork freelance jobs. 
The pipeline includes:
1. **Data Cleaning:** Removing nulls, duplicates, and short irrelevant descriptions.
2. **Semantic Classification:** Using NLP (`SentenceTransformers`) to classify jobs into specific technical categories based on semantic similarity to extract the job roles on our platform.
3. **Language Detection:** Filtering the dataset to keep only English job postings.

## 1. Importing Libraries & Loading Data
Loading the necessary libraries and importing the Upwork dataset to take an initial look at its structure.

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch

In [2]:
df = pd.read_csv('/kaggle/input/datasets/ahmedmyalo/upwork-freelance-jobs-60k/Final_Upwork_Dataset.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63950 entries, 0 to 63949
Data columns (total 41 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Job Title             63948 non-null  object 
 1   Job_URL               63949 non-null  object 
 2   EX_level_demand       56935 non-null  object 
 3   Time_Limitation       23034 non-null  object 
 4   Search_Keyword        63949 non-null  object 
 5   Posted_from           63913 non-null  object 
 6   Description           63948 non-null  object 
 7   Category1_URL_search  63505 non-null  object 
 8   Category_1            63505 non-null  object 
 9   highlight             50131 non-null  object 
 10  Category2_URL_search  59889 non-null  object 
 11  Category_2            59889 non-null  object 
 12  Category3_URL_search  54997 non-null  object 
 13  Category_3            54997 non-null  object 
 14  Category4_URL_search  47543 non-null  object 
 15  Category_4         

## 2. Data Cleaning & Preprocessing
In this section, we will:
* Keep only the essential columns (`Job Title` and `Description`).
* Handle missing values and drop duplicated entries.
* Filter out jobs with very short descriptions (less than 30 characters) to ensure data quality.

In [4]:
df = df[['Job Title','Description']]

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63950 entries, 0 to 63949
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Job Title    63948 non-null  object
 1   Description  63948 non-null  object
dtypes: object(2)
memory usage: 999.3+ KB


In [6]:
total_nulls = df.isnull().sum().sum()
total_nulls

np.int64(4)

In [7]:
df.duplicated().sum()

np.int64(4060)

In [8]:
df = df.dropna()

In [9]:
df = df.drop_duplicates()

In [10]:
df['Description'].str.len().describe()

count    59887.000000
mean       646.425535
std        680.027897
min          1.000000
25%        207.000000
50%        399.000000
75%        832.000000
max       5000.000000
Name: Description, dtype: float64

In [11]:
df= df[df['Description'].str.len() >= 30]

In [13]:
len(df)

59857

## 3. Semantic Job Classification (NLP)
Instead of relying on basic keyword matching, we use the `all-MiniLM-L6-v2` model from **SentenceTransformers** to compute cosine similarity between the job descriptions and predefined technical categories. 

Jobs with a similarity score `> 0.4` are kept and assigned to their best-matching category.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

target_categories = {
    'AI Engineer': 'AI Engineer, Machine Learning, Deep Learning, NLP, Computer Vision',
    'Frontend Developer': 'Frontend Developer, React, Angular, Vue, CSS, UI Development',
    'Backend Developer': 'Backend Developer, Node.js, Python, Django, API, SQL, Java',
    'DevOps Engineer': 'DevOps Engineer, AWS, Docker, Kubernetes, CI/CD, Cloud Infrastructure',
    'Data Analyst': 'Data Analyst, Power BI, Tableau, SQL, Data Visualization, Excel Analysis',
    'Mobile Developer': 'Mobile Developer, iOS, Android, Flutter, React Native, Swift',
    'Content Writer': 'Content Writer, Copywriting, Blog Post, SEO Writing, Article Writer',
    'Video Producer': 'Video Producer, Video Editor, Motion Graphics, After Effects',
    'Graphic Designer': 'Graphic Designer, Photoshop, Illustrator, Branding, Logo Design'
}

category_names = list(target_categories.keys())
category_descriptions = list(target_categories.values())
category_embeddings = model.encode(category_descriptions, convert_to_tensor=True)


df['text_to_search'] = df['Job Title'].fillna('') + " " + df['Description'].fillna('').str[:100]

job_embeddings = model.encode(df['text_to_search'].tolist(), batch_size=64, show_progress_bar=True, convert_to_tensor=True)

cosine_scores = util.cos_sim(job_embeddings, category_embeddings)


max_scores, category_indices = torch.max(cosine_scores, dim=1)

df['best_category'] = [category_names[i] for i in category_indices]
df['similarity_score'] = max_scores.tolist()

final_df = df[df['similarity_score'] > 0.4]

output = final_df[['Job Title', 'Description', 'best_category']]
output.to_csv('ai_smart_filtered_jobs.csv', index=False)

print(f"Final Data Length:{len(output)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/936 [00:00<?, ?it/s]

Final Data Length:11355


In [15]:
stats = final_df.groupby('best_category').agg({
    'Job Title': 'count',
}).rename(columns={'Job Title': 'Job Count', 'quality_score': 'Avg Quality', 'similarity_score': 'Avg Relevance'})

stats

,Job Count
best_category,
AI Engineer,436
Backend Developer,1614
Content Writer,1721
Data Analyst,1152
DevOps Engineer,250
Frontend Developer,2081
Graphic Designer,1312
Mobile Developer,1986
Video Producer,803


In [16]:
final_df[['Job Title', 'Description', 'best_category']]

,Job Title,Description,best_category
2,"File Maker Pro Reports, Charts, Query and Ongo...",NITIAL PROJECT\n\nSet up Monthly Report mimick...,Data Analyst
4,BI and Data Engineer for Upwork Finance System...,The Upwork Finance Systems team is looking for...,Data Analyst
5,Computer vision / machine learning: synthetic ...,Our company is working on a government proposa...,Video Producer
6,Expert in Data Analytics,Mission : Ensure smooth transition of Analytic...,Data Analyst
7,Data Analytics Expert for Educational Exercise,Looking for someone to help me solve some prob...,Data Analyst
...,...,...,...
63920,"SEO, landing Page, Google Control Pane and ana...",Complete points 11 & 13 from the word document...,Content Writer
63930,SEO content writer for articles and blog posts.,I need a content writer to assist me with my d...,Content Writer
63937,Graphic designer needed for various projects i...,Design support for five US-based food brands i...,Graphic Designer
63940,SEO B2B content writer in German,"Auf die Plätze, fertig, SEO! Wir suchen nach e...",Content Writer


## 4. Language Detection & Final Filtering
Since we are focusing on English text, we use the `langdetect` library to filter out non-English job postings. To optimize performance, the detection is applied only to the first 200 characters of each description.

In [17]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.7 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=0b30f3cf4c1d02f25f77de51346dd445d8b6ce0afb7c4f03db223fbf814a6767
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

DetectorFactory.seed = 0

def is_english(text):
    try:
        sample = str(text)[:200].strip()
        if len(sample) < 10: 
            return False
        return detect(sample) == 'en'
    except Exception:
        return False


final_df['is_en'] = final_df['Description'].apply(is_english)

english_jobs_df = final_df[final_df['is_en'] == True].copy()


final_output = english_jobs_df[['Job Title', 'Description', 'best_category']]

final_output.to_csv('jobs_5_data.csv', index=False)

print(f"Final Data Length:{len(final_output)}")

/tmp/ipykernel_55/1160346705.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['is_en'] = final_df['Description'].apply(is_english)


Final Data Length:11282
